In [27]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 
from core.model import ME_Model

from utils import functions as func
from utils import parameters as params
from utils import metabolites as metab

from tqdm import tqdm
import copy
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/other/test_lp/'

In [28]:
base = 0
counter = 3
mu_val = 1e-9

In [29]:
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme1 = pickle.load(handle)
with open(lp_path + 'working_version_' + str(base) + '.pickle', 'rb') as handle:
    tme00 = pickle.load(handle)


S_1 = tme1.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
S_0 = tme00.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')

mapper = {'125250296_complex_n': '191091342_complex_n'}
S_0.index = [col if col not in mapper else mapper[col] for col in S_0.index.tolist()]


S_0 = S_0.loc[S_1.index, S_1.columns]
mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))



Does changing to original stoichiometry make model feasible? Yes

Remember, the tme0 that works had incorrect mass balance on degradation of ribosome

In [31]:
tme0 = copy.deepcopy(tme1)
for coord in mismatch:
    rxn_id = S_0.columns.tolist()[coord[1]]
    metab_id = S_0.index.tolist()[coord[0]]
    tme0.reactions.get_by_id(rxn_id)._metabolites[tme0.metabolites.get_by_id(metab_id)] = S_0.iloc[tuple(coord)]

sln0, stat0, _ = tme0.solve_lp(mu_val = 0.03)
sln1, stat1, _ = tme1.solve_lp(mu_val = 0.03)


fluxes = pd.DataFrame(data = {'feasible_original': sln0[:len(tme0.reactions)]}, 
                     index = [r.id for r in tme0.reactions])
fluxes['infeasible'] = [sln1[tme1.reactions.index(rxn_id)] for rxn_id in fluxes.index]

Getting MINOS parameters...
Done in 86.7981 seconds with status 0
Getting MINOS parameters...
Done in 120.485 seconds with status 1


Which of the original reaction stoichiometries influences infeasibility?

In [51]:
rxn_indeces = set([i[1] for i in mismatch])
changed_reactions = []
for ri in tqdm(rxn_indeces):
    test_model = copy.deepcopy(tme1)
    rxn_id = S_0.columns.tolist()[ri]
    metab_indeces = [i[0] for i in mismatch if i[1] == ri]
    for mi in metab_indeces:
        metab_id = S_0.index.tolist()[mi]
        test_model.reactions.get_by_id(rxn_id)._metabolites[test_model.metabolites.get_by_id(metab_id)] = S_0.iloc[mi, ri]
    sln, stat, _ = test_model.solve_lp(mu_val = 1e-9)
    if stat.reshape(1)[0] == 0:
        changed_reactions.append(rxn_id)

changed_reactions

['TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc',
 'co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc']

In [84]:
ri = 7392 # TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc

test_model = copy.deepcopy(tme1)
rxn_id = S_0.columns.tolist()[ri]
metab_indeces = [i[0] for i in mismatch if i[1] == ri]
for mi in metab_indeces:
    metab_id = S_0.index.tolist()[mi]
    test_model.reactions.get_by_id(rxn_id)._metabolites[test_model.metabolites.get_by_id(metab_id)] = S_0.iloc[mi, ri]
sln_test, stat_test, _ = test_model.solve_lp(mu_val = 0.03)
fluxes['feasible_ribosomaldeg'] = [sln_test[test_model.reactions.index(rxn_id)] for rxn_id in fluxes.index]

Getting MINOS parameters...
Done in 116.227 seconds with status 0


In [65]:
m0 = {m.id: c for m,c  in tme0.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').metabolites.items()}
m1 = {m.id: c for m,c  in tme1.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').metabolites.items()}
m0 = {m_id: c for m_id,c in m0.items() if m1[m_id] != c}
m1 = {m_id: c for m_id,c in m1.items() if m_id in m0}



In [66]:
fluxes.loc[changed_reactions,:]


,feasible,infeasible,feasible_single
TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc,0.000004,0.0,0.000004
co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc,0.000000,0.0,0.000000


In [67]:
m0

{'h2o_c': -119.0,
 'atp_c': 1.0,
 'h_c': 119.0,
 'cmp_c': 29.0,
 'ump_c': 28.0,
 'gmp_c': 33.0,
 'amp_c': 29.0}

In [68]:
m1

{'h2o_c': -31141,
 'atp_c': -8005,
 'h_c': 15221,
 'cmp_c': 2268,
 'ump_c': 1219,
 'gmp_c': 2453,
 'amp_c': 1275}

It looks like the model wants to creat SOME nmp_c's (flux through feasible), but not at the cost of lots of atp/h2o consumption. We can test this by adding ATP and nmp's as boundary reactions:

In [76]:
exchange_metabs = []
for met_id in tqdm(['atp_c', 'cmp_c', 'ump_c', 'gmp_c', 'amp_c']):
    tme_test = copy.deepcopy(tme1)
    met_obj = tme_test.metabolites.get_by_id(met_id)
    tme_test.add_boundary(met_obj, type = 'sink')
    sln, stat, _ = tme_test.solve_lp(mu_val = 0.03)
    if stat.reshape(1)[0] == 0:
        exchange_metabs.append(met_id)
exchange_metabs

['cmp_c', 'gmp_c']

Oddly enough, adding an exchange for ATP does not help with feasibility, but adding one for the CMP/GMP does. So, it is not simpy a tradeoff b/w ATP and NMPs. 

Note that this parallels the feasibility issues encountered by coupling degradation of ribosome to translation, which makes sense. 

In [80]:
tme_exchange = copy.deepcopy(tme1)
met_obj = tme_exchange.metabolites.get_by_id('cmp_c')
tme_exchange.add_boundary(met_obj, type = 'sink')
sln_exchange, stat, _ = tme_exchange.solve_lp(mu_val = 0.03)
fluxes['feasible_cmpcexch'] = [sln_exchange[tme_exchange.reactions.index(rxn_id)] for rxn_id in fluxes.index]

Reassign .coupled_metabolites attribute
Getting MINOS parameters...
Done in 110.505 seconds with status 0


In [95]:
fluxes = fluxes.iloc[:, :4]

In [88]:
tme_exchange.reactions.get_by_id('SK_cmp_c').reaction

'cmp_c <=> '

In [89]:
sln_exchange[tme_exchange.reactions.index('SK_cmp_c')]

0.2216451394632891

In [145]:
# get all the RNA reactiosn in the model

[]

In [ ]:
-->mrna + (MW(mrna))*biomass_mrna

biomass_mrna --> biomass
biomass --> # dilution reaction

# Option 1
A --> B + proxy
C + c*proxy --> D

# Option 2
A + B + c*C --> c*D

#################################################################
--> mRNA # formation
mRNA --> NTPs # degradation
aa --> portein # protein synthesis


aa + (c1+c2)*mrna --> protein + c2*NTPs # still is not mass balanced

#################################################################
mRNA --> NTPs + deg_proxy # degradation

aa + c1*mRNA + c2*deg_proxy --> protein # protein synthesis

--> mRNA + form_proxy # formation
aa + f(mu)*mRNA + k*form_proxy + c2*deg_proxy --> protein # protein synthesis
c1 = f(mu) + k
#################################################################


In [131]:
.ribosome_biogenesis = True
.sink = True 
.coupled_metabolites

Reaction identifier,mRNA_biomass_to_biomass
Name,
Memory address,0x07f733ab7b860
Stoichiometry,biomass_mRNA --> biomass -->
GPR,
Lower bound,0.0
Upper bound,1000.0


In [144]:
[m for m in tme1.metabolites if ('premRNA' in m.id)]

[<Biomass biomass_premRNA at 0x7f733ab7b9e8>]

In [ ]:
tme

The exchange reaction proceeds in the forward direction, further indicating that excess CMP is being created

We can further test this by seeing which of the metabolite stoichiometries specifically in the reaction matters most

In [101]:
stoich_metabs = []

ri = 7392 # TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc

rxn_id = S_0.columns.tolist()[ri]
metab_indeces = [i[0] for i in mismatch if i[1] == ri]
for mi in tqdm(metab_indeces):
    tme_test = copy.deepcopy(tme1)
    metab_id = S_0.index.tolist()[mi]
    tme_test.reactions.get_by_id(rxn_id)._metabolites[tme_test.metabolites.get_by_id(metab_id)] = S_0.iloc[mi, ri]
    sln_test, stat_test, _ = tme_test.solve_lp(mu_val = 0.03)
    if stat_test.reshape(1)[0] == 0:
        stoich_metabs.append(metab_id)
        fluxes['feasible_stoich_' + metab_id] = [sln_test[tme_test.reactions.index(rxn_id)] for rxn_id in fluxes.index]
        
        
stoich_metabs

['cmp_c', 'gmp_c']

In [102]:
fluxes.loc[changed_reactions, :]

,feasible_original,infeasible,feasible_cmpcexch,feasible_ribosomaldeg,feasible_stoich_cmp_c,feasible_stoich_gmp_c
TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc,0.000004,0.0,0.0,0.000004,0.000009,0.000008
co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc,0.000000,0.0,0.0,0.000000,0.000000,0.000000


In [106]:
tme1.infeasible_reactions(mu_val = 0.03, sln = sln1, stat = stat1)

{'r0707_R': 3.711888560523877e-18,
 'COPII_IMPORTtg_COMPLEX_DEUBIQUITINATIONc': 8.402521461111328e-18}

In [114]:
m_ir = {m.id: c for m,c in tme1.reactions.get_by_id('COPII_IMPORTtg_COMPLEX_DEUBIQUITINATIONc').metabolites.items()}
m_cr = {m.id: c for m,c in tme1.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').metabolites.items()}
isct = set(m_ir).intersection(m_cr)
m_ir = {m:c for m,c in m_ir.items() if m in isct}
m_cr = {m:c for m,c in m_cr.items() if m in isct}

In [118]:
m_ir

{'h2o_c': -1,
 'cleaved_polyubiquitin_moiety_protein_c': 1,
 'DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c': -9.87244365228048e-6*mu - 1.67158859837851e-7,
 'DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy': -1.671588598378511e-07,
 'biomass_protein': 0.0180152799999505}

In [119]:
m_cr

{'h2o_c': -31141,
 'cleaved_polyubiquitin_moiety_protein_c': 1,
 'DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c': -9.87244365228048e-6*mu - 1.67158859837851e-7,
 'DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy': -1.671588598378511e-07,
 'biomass_protein': -1814.87689370000}

In [121]:
tme1.reactions.get_by_id('COPII_IMPORTtg_COMPLEX_DEUBIQUITINATIONc').reaction

'COPII_IMPORTtg_polyub_complex_c + 1.671588598378511e-07 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy + 9.87244365228048e6*mu  1.67158859837851e7 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c + h2o_c --> COPII_IMPORTtg_complex_c + 0.0180152799999505 biomass_protein + cleaved_polyubiquitin_moiety_protein_c'

In [122]:
tme1.reactions.get_by_id('COPII_IMPORTtg_COMPLEX_DEUBIQUITINATIONc').coupled_metabolites

{<Complex DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c at 0x7f7338406240>: 'catalysis',
 <Macromolecule DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy at 0x7f7338406da0>: 'enzyme_degradation'}

In [123]:
tme1.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').coupled_metabolites

{<Complex DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c at 0x7f7338406240>: 'catalysis',
 <Complex 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_complex_c at 0x7f733e2af7f0>: 'catalysis',
 <Macromolecule 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_COMPLEX_enzyme_deg_proxy at 0x7f733e2aff60>: 'enzyme_degradation',
 <Macromolecule DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy at 0x7f7338406da0>: 'enzyme_degradation'}

In [126]:
[r for r in tme1.reactions if 'DEUBIQUITINATIONc' in r.id][0].coupled_metabolites

{<Complex DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c at 0x7f7338406240>: 'catalysis',
 <Macromolecule DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy at 0x7f7338406da0>: 'enzyme_degradation'}

In [127]:
[r for r in tme1.reactions if 'PROTEASOMAL_DEGRADATIONc' in r.id][0].coupled_metabolites

{<Complex DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c at 0x7f7338406240>: 'catalysis',
 <Macromolecule DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy at 0x7f7338406da0>: 'enzyme_degradation'}

Why is DEUBIQUITINATIONc and PROTEASOMAL_DEGRADATIONc getting the same enzyme?